# OCONUS Impervious Data

This notebook documents the steps used to derive 250 m percent impervious rasters for southern Alaska, Hawaii, Puerto Rico / Virgin Islands

Data Source: NOAA CCAP 2020 - 2021 1 meter Impervious data downloaded January 2026 for Hawaii, Puerto Rico, Virgin Islands, Alaska

- Bulk download: https://coastalimagery.blob.core.windows.net/ccap-landcover/CCAP_bulk_download/High_Resolution_Land_Cover/Phase_1_Initial_Layers/Impervious/index.html
- About: https://coast.noaa.gov/digitalcoast/data/ccaphighres.html 

Each region was downloaded, resampled to 250 m resolution using 'SUM' method, and divided by total area of grid cell to derive percent impervious at new resolution.

Files were merged where needed:
- All Hawaiian islands reprojected to 32604 and merged
- PRVI merged
- AK merged after many smaller pieces created

Final layers stored: 
s3://edfs-data/attributes/{projection}/impervious/{region}_full_imperv_percent.tif


In [ ]:
import os

import numpy as np
import xarray as xr
from rasterio.enums import Resampling
from rasterio.merge import merge

## Hawaii and PRVI
Hawaii was separated into 4 files. Puerto Rico and Virgin Islands were two separate files. Each file was processed as the sample shows below and merged for final domain.

In [ ]:
# Example worfklow for single layer - Do not run
ccap = "./data/hi_2021_ccap_v2_hires_impervious_20240206/hi_kauai_2021_ccap_v2_hires_impervious_20240206.tif"
output = "kauai"
raster = xr.open_dataarray(ccap, engine="rasterio", chunks="auto")
raster = raster.astype(np.uint16)
sum_raster = raster.rio.reproject(raster.rio.crs, resolution=250, resampling=Resampling.sum).astype(np.uint16)
sum_raster.rio.to_raster(
    f"../data/hi/hi_{output}_imperv_sum.tif", tiled="YES", compress="deflate", driver="GTiff"
)
percent_raster = (sum_raster / (250 * 250) * 100).round(1).astype(np.uint16)
percent_raster.rio.to_raster(
    f"../data/hi/hi_imperv_{output}_percent.tif", tiled="YES", compress="deflate", driver="GTiff"
)


In [ ]:
# Sample merge - Do not run
hi_output = ["hono", "maui", "kauai", "hawaii"]
merge_files = [f"../data/hi/hi_imperv_{out}_percent.tif" for out in hi_output]
merge(
    merge_files,
    res=250,
    dtype=np.uint8,
    resampling=Resampling.bilinear,
    dst_path="../data/hi/hi_full_imperv_percent.tif",
    dst_kwds={"compress": "lzw", "tiled": "YES"},
)


## Alaska

The state of Alaska was served in multiple batches and each batch had multiple tiles. The southern Alaska domain used:
- ak_2020_ccap_v2_hires_impervious_interior_20240112
- ak_2020_ccap_v2_hires_impervious_southcentral_22040112

Within these folders, the following tiles covered the domain: 7, 8, 12, 13, 14, 21, 22, 23, 24 

At 1 meter resolution, tiles were too large to be processed individually. They were broken into smaller pieces manually in QGIS with some trial and error to get efficient sizes and eliminate processing over non-domain regions.

All pieces tiles were resampled to 250 meter for percent impervious, merged, and reprojected from 5070 to 3336. 

Note that when viewing the layer, QGIS's default output does not compute stats correctly due to large, sparse raster. To visualize correctly, set min/max range 0-100.

In [ ]:
# Sample loop used - Do not run
ccap = "../data/ak"
aks = ["alaska_mosaic_dataset22_f.tif", "alaska_mosaic_dataset22_e.tif", "alaska_mosaic_dataset22_d.tif"]

output = ["ak_22_f", "ak_22_e", "ak_22_d"]

for i in range(len(aks)):
    f = os.path.join(ccap, aks[i])
    output_f = output[i]
    raster = xr.open_dataarray(f, engine="rasterio", chunks="auto")
    raster = raster.astype(np.uint16)
    sum_raster = raster.rio.reproject(raster.rio.crs, resolution=250, resampling=Resampling.sum).astype(
        np.uint16
    )
    sum_raster.rio.to_raster(
        f"../data/ak/{output_f}_imperv_sum.tif", tiled="YES", compress="deflate", driver="GTiff"
    )
    percent_raster = (sum_raster / (250 * 250) * 100).round(1).astype(np.uint16)
    percent_raster.rio.to_raster(
        f"../data/ak/{output_f}_imperv_percent.tif", tiled="YES", compress="deflate", driver="GTiff"
    )
    del raster, sum_raster, percent_raster
    print(i)